# 00 · The simulated experiment and its planted truth

**What this notebook shows.** How 5,000 institutional clients, their fee tiers and a 13-week
multi-cell incentive experiment are simulated, and what the *planted truth* (every client's
potential outcomes under every cell) looks like. Everything downstream is graded against this truth.

Read with: [`docs/learn/01_potential_outcomes.md`](../docs/learn/01_potential_outcomes.md),
[`02_randomisation_and_design.md`](../docs/learn/02_randomisation_and_design.md) and
[`docs/SIMPLIFICATIONS.md`](../docs/SIMPLIFICATIONS.md). Numbers come from `configs/default.yaml`.

In [1]:
import json, warnings
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import Image, display
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30); pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
from tierlift.config import REPO_ROOT, load_config
from tierlift.simulate import SimulatedData
cfg = load_config()
DATA, RESULTS, FIGS = REPO_ROOT / "data", REPO_ROOT / "results", REPO_ROOT / "reports" / "figures"
assert DATA.exists(), "run `make all` (or `tierlift run-all`) first: the notebooks read data/ and results/"
data = SimulatedData.load(DATA)
e, truth = data.experiment, data.truth.set_index("client_id")
primary, control = cfg.analysis.primary_contrast
def fig(name): display(Image(filename=str(FIGS / name), width=900))
print(f"seed {cfg.seed}, {len(e):,} clients, cells {cfg.experiment.cells}")

seed 13, 5,000 clients, cells {'holdout': 0.3, 'nudge': 0.3, 'incentive': 0.4}


## 1. The fee schedule and where clients sit relative to the next tier

In [2]:
from tierlift.fees import FeeSchedule
fees = FeeSchedule(cfg.tiers)
pd.DataFrame([{"tier": t.name, "min_quarterly_volume_$": t.min_volume, "maker_bps": t.maker_bps, "taker_bps": t.taker_bps} for t in cfg.tiers])

,tier,min_quarterly_volume_$,maker_bps,taker_bps
0,Starter,0,8.0000,14.0000
1,Standard,25000000,7.0000,12.0000
2,Professional,100000000,6.0000,10.5000
3,Enterprise,500000000,5.0000,9.0000


In [3]:
clients = data.clients
print(clients.shape); display(clients.head())
display(pd.crosstab(clients["tier"], clients["distance_bucket"], margins=True))

(5000, 17)


,client_id,segment,region,tenure_months,has_account_manager,api_share,dashboard_logins_pm,maker_ratio,weekly_cv,pre_volume,pre_revenue,tier_idx,tier,distance_pct,distance_bucket,engagement_z,aum_flow_z
0,1,Asset Manager,Americas,31,0,0.3744,4,0.3119,0.2342,"67,860,893.9437","70,850.4996",1,Standard,0.3214,20_plus,0.4191,-0.4400
1,2,Asset Manager,Americas,15,0,0.3290,3,0.3358,0.1644,"44,712,337.4645","46,146.6476",1,Standard,0.5529,20_plus,0.0526,0.6566
2,3,Asset Manager,Americas,42,0,0.3480,2,0.3612,0.2831,"35,295,987.7721","35,981.1235",1,Standard,0.6470,20_plus,-0.4199,-1.4274
3,4,Market Maker,EMEA,82,0,0.9148,1,0.8030,0.1239,"143,609,816.8241","98,896.3699",2,Professional,0.7128,20_plus,-1.0859,0.5360
4,5,Hedge Fund,APAC,64,1,0.4404,5,0.4577,0.4992,"381,430,564.0375","321,935.6631",2,Professional,0.2371,20_plus,0.7186,0.1616


distance_bucket,10_to_20,20_plus,top_tier,within_10,All
tier,,,,,
Enterprise,0,0,11,0,11
Professional,12,866,0,9,887
Standard,193,2507,0,165,2865
Starter,203,862,0,172,1237
All,408,4235,11,346,5000


`distance_pct` is how far a client's *pre-period* quarterly volume is below the next threshold.
The 10–20% band is the one the business case will target; within-10% clients would cross anyway and
20%+ clients would need an implausible lift.

## 2. Stratified randomisation into three cells

In [4]:
a = data.assignment
display(a["cell"].value_counts(normalize=True).rename("share").to_frame().T)
print("strata:", a["stratum"].nunique(), "| cells are allocated in permuted blocks inside each stratum")
display(pd.crosstab(e["distance_bucket"], e["cell"], normalize="index").round(3))

cell,incentive,holdout,nudge
share,0.3998,0.3006,0.2996


strata: 52 | cells are allocated in permuted blocks inside each stratum


cell,holdout,incentive,nudge
distance_bucket,,,
10_to_20,0.2990,0.4040,0.2970
20_plus,0.3000,0.4000,0.3000
top_tier,0.2730,0.3640,0.3640
within_10,0.3090,0.3960,0.2950


## 3. The planted truth: potential outcomes under every cell

Only the simulator sees this table. The analysis code sees `experiment` (one realised outcome per client) and never reads `truth`.

In [5]:
display(truth.head())
lift = truth.groupby(e.set_index("client_id")["distance_bucket"])[["tau_volume_incentive", "tau_volume_nudge"]].mean()
display(lift.rename(columns={"tau_volume_incentive": "true mean volume lift (incentive)", "tau_volume_nudge": "nudge"}).style.format("{:.1%}"))
print("planted lifts by distance (config):", cfg.truth.incentive_lift_by_distance)

,drift,tau_volume_incentive,tau_volume_nudge,lapse_time,weekly_hazard,volume_holdout,volume_nudge,volume_incentive,rate_holdout,rate_nudge,rate_incentive,revenue_holdout,revenue_nudge,revenue_incentive,crossed_holdout,crossed_nudge,crossed_incentive,ite_revenue_nudge,ite_volume_nudge,ite_revenue_incentive,ite_volume_incentive
client_id,,,,,,,,,,,,,,,,,,,,,
1,0.6402,0.0602,0.0150,4.4627,0.0191,"45,650,485.3822","45,921,633.3141","46,735,077.1098",0.0010,0.0011,0.0010,"47,661.3893","48,633.3058","46,319.0181",False,False,False,971.9165,"271,147.9319","-1,342.3712","1,084,591.7276"
2,1.0752,0.0240,0.0060,103.3756,0.0198,"45,801,363.5300","46,076,162.6408","46,900,559.9733",0.0010,0.0010,0.0010,"47,271.5873","48,246.3499","45,950.5891",False,False,False,974.7626,"274,799.1108","-1,320.9982","1,099,196.4433"
3,0.4403,0.0360,0.0090,3.2029,0.0214,"16,247,054.5367","16,298,141.9253","16,451,404.0914",0.0010,0.0010,0.0010,"16,562.2474","16,858.7980","15,919.6947",False,False,False,296.5506,"51,087.3887",-642.5527,"204,349.5547"
4,1.0988,0.0054,0.0013,22.2311,0.0296,"169,358,980.5592","169,586,149.7196","170,267,657.2010",0.0007,0.0007,0.0006,"116,629.0620","119,074.9150","110,073.7837",False,False,False,"2,445.8531","227,169.1604","-6,555.2783","908,676.6418"
5,0.8595,0.0611,0.0153,2.4363,0.0133,"324,230,985.0620","325,350,267.0155","328,708,112.8761",0.0008,0.0009,0.0008,"273,662.2995","278,999.2412","260,740.3146",False,False,False,"5,336.9417","1,119,281.9535","-12,921.9849","4,477,127.8141"


,true mean volume lift (incentive),nudge
distance_bucket,,
10_to_20,32.7%,8.2%
20_plus,3.8%,1.0%
top_tier,0.0%,0.0%
within_10,8.2%,2.1%


planted lifts by distance (config): {'within_10': 0.08, '10_to_20': 0.3, '20_plus_near': 0.06, '20_plus_far': 0.03, 'top_tier': 0.0}


In [6]:
rel = {c: truth[f"revenue_{c}"].sum() / truth["revenue_holdout"].sum() - 1 for c in cfg.treatment_cells}
print("true population revenue effect of a blanket rollout:", {k: f"{v:+.2%}" for k, v in rel.items()})
band = e["distance_bucket"].eq("10_to_20").to_numpy()
tb = truth.loc[e.loc[band, "client_id"]]
print("true revenue effect on the 10-20% band:", f"{tb['revenue_incentive'].sum() / tb['revenue_holdout'].sum() - 1:+.2%}")

true population revenue effect of a blanket rollout: {'nudge': '+2.81%', 'incentive': '-2.97%'}
true revenue effect on the 10-20% band: +11.83%


## 4. What the analyst actually sees

In [7]:
display(e.head())
print("columns that never appear in the observed tables:", [c for c in truth.columns if c in e.columns])
display(data.weekly.query("client_id == @e.client_id.iloc[0]").head(8))

,client_id,segment,region,tenure_months,has_account_manager,api_share,dashboard_logins_pm,maker_ratio,weekly_cv,pre_volume,pre_revenue,tier_idx,tier,distance_pct,distance_bucket,engagement_z,aum_flow_z,cell,stratum,exp_volume,exp_revenue,exp_rate,crossed_tier,lapse_week,lapsed
0,1,Asset Manager,Americas,31,0,0.3744,4,0.3119,0.2342,"67,860,893.9437","70,850.4996",1,Standard,0.3214,20_plus,0.4191,-0.4400,incentive,Standard|20_plus|Asset Manager,"46,735,077.1098","46,319.0181",0.0010,0,5.0000,1
1,2,Asset Manager,Americas,15,0,0.3290,3,0.3358,0.1644,"44,712,337.4645","46,146.6476",1,Standard,0.5529,20_plus,0.0526,0.6566,nudge,Standard|20_plus|Asset Manager,"46,076,162.6408","48,246.3499",0.0010,0,39.0000,0
2,3,Asset Manager,Americas,42,0,0.3480,2,0.3612,0.2831,"35,295,987.7721","35,981.1235",1,Standard,0.6470,20_plus,-0.4199,-1.4274,incentive,Standard|20_plus|Asset Manager,"16,451,404.0914","15,919.6947",0.0010,0,4.0000,1
3,4,Market Maker,EMEA,82,0,0.9148,1,0.8030,0.1239,"143,609,816.8241","98,896.3699",2,Professional,0.7128,20_plus,-1.0859,0.5360,nudge,Professional|20_plus|Market Maker,"169,586,149.7196","119,074.9150",0.0007,0,23.0000,1
4,5,Hedge Fund,APAC,64,1,0.4404,5,0.4577,0.4992,"381,430,564.0375","321,935.6631",2,Professional,0.2371,20_plus,0.7186,0.1616,incentive,Professional|20_plus|Hedge Fund,"328,708,112.8761","260,740.3146",0.0008,0,3.0000,1


columns that never appear in the observed tables: []


,client_id,week,volume
0,1,1,"5,230,168.0172"
1,1,2,"4,224,067.0290"
2,1,3,"3,633,590.2495"
3,1,4,"2,739,559.0201"
4,1,5,"3,275,563.0078"
5,1,6,"4,193,683.3838"
6,1,7,"2,803,548.7713"
7,1,8,"3,094,544.5538"


## 5. Data-quality gates

`run_all_checks` runs before any analysis; blocking failures stop the pipeline.

In [8]:
from tierlift.validation.checks import run_all_checks
display(run_all_checks(data, cfg))

,passed,detail,blocking
name,,,
unique clients,True,"5000 rows, 0 duplicates",True
sample ratio mismatch,True,"chi2 p = 0.995, counts {'holdout': 1503, 'nudg...",True
cell shares within strata,True,worst deviation from plan 0.027,True
covariate balance,True,max |SMD| = 0.062 across 16 cell x covariate p...,False
revenue = volume x rate,True,max abs error $0.00e+00,True
observed outcome = potential outcome of assigned cell,True,max abs error 0.00e+00,True
weekly panel sums to quarter volume,True,max relative error 3.40e-15,True
no post-treatment features,True,"19 features, leaked: []",True
no treatment leakage into pre-period,True,relative spread of pre-period means 0.008,False


## Check your understanding

1. Why is `distance_pct` computed from *pre-period* volume and not from the experiment quarter?
2. Two clients are identical except one is in `holdout` and one in `incentive`. Which columns of `truth` are the same for both, and which differ?
3. If assignment were *not* stratified, which figure in the next notebook would change first?